# Blackbody Radiation and the Forge of Vulcan
## Pedagogical notebook for instructors

This notebook accompanies the article:
> Galeano J R (2026) *Art and Physics at the Museo del Prado: A Walk Through Science Hidden in Masterpieces*. European Journal of Physics.

It supports the pedagogical activity in **Section 4 (Velázquez)**, Block 1: *Colour as a thermometer*.

The notebook answers the three questions posed in the article:
1. Is the visible power emitted by the forge billet at 900 °C detectable by the human eye?
2. At what temperature does the fraction of visible emission reach 1%?
3. Which wavelength dominates within the visible range at 900 °C, and is it consistent with the orange-red colour depicted by Velázquez?

**Intended use:** This notebook is designed as a reference for instructors. Students are encouraged to write their own code guided by the article, and to compare their results with those produced here.

## 0. Imports and physical constants

We use `scipy.constants` for all physical constants, ensuring maximum precision and avoiding manual transcription errors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq
import scipy.constants as const

# Physical constants from scipy.constants
h    = const.h        # Planck constant (J·s)
c    = const.c        # Speed of light in vacuum (m/s)
k_B  = const.k        # Boltzmann constant (J/K)
sigma_SB = const.sigma  # Stefan-Boltzmann constant (W/m²/K⁴)

print(f"Planck constant      h    = {h:.6e} J·s")
print(f"Speed of light       c    = {c:.6e} m/s")
print(f"Boltzmann constant   k_B  = {k_B:.6e} J/K")
print(f"Stefan-Boltzmann     σ_SB = {sigma_SB:.6e} W/m²/K⁴")

# Wien displacement constant (derived)
b_wien = const.Wien
print(f"Wien displacement    b    = {b_wien:.6e} m·K")

## 1. Planck spectral radiance and Wien's displacement law

The Planck spectral radiance $B(\lambda, T)$ gives the power emitted per unit surface area per unit wavelength interval:

$$B(\lambda, T) = \frac{2hc^2}{\lambda^5} \frac{1}{e^{hc/\lambda k_B T} - 1}$$

The peak wavelength is given by Wien's displacement law:

$$\lambda_{\max} = \frac{b}{T}$$

In [ ]:
def planck(lam, T):
    """
    Planck spectral radiance B(lambda, T) in W/m²/m.
    
    Parameters
    ----------
    lam : float or array
        Wavelength in metres.
    T : float
        Temperature in kelvin.
    
    Returns
    -------
    float or array
        Spectral radiance in W/m²/m.
    """
    exponent = h * c / (lam * k_B * T)
    return (2 * h * c**2 / lam**5) / (np.exp(exponent) - 1)

def wien_peak(T):
    """Peak wavelength from Wien's displacement law, in nm."""
    return b_wien / T * 1e9  # convert m to nm

# Example: forge temperature
T_forge_C = 900          # degrees Celsius
T_forge   = T_forge_C + 273.15  # kelvin

lam_peak = wien_peak(T_forge)
print(f"Forge temperature: T = {T_forge_C} °C = {T_forge:.2f} K")
print(f"Wien peak wavelength: λ_max = {lam_peak:.1f} nm")
print(f"(Visible range: 380–700 nm → peak is in the near infrared)")

## 2. Fraction of visible emission $f_{\rm vis}(T)$

The fraction of total blackbody power emitted in the visible range (380–700 nm) is:

$$f_{\rm vis}(T) = \frac{\displaystyle\int_{380\,{\rm nm}}^{700\,{\rm nm}} B(\lambda,T)\,d\lambda}{\sigma_{\rm SB}\,T^4}$$

The denominator is the Stefan–Boltzmann law (analytic). The numerator is computed numerically.

In [ ]:
# Visible range limits
lam_vis_min = 380e-9   # 380 nm in metres
lam_vis_max = 700e-9   # 700 nm in metres

def f_vis(T):
    """
    Fraction of blackbody power emitted in the visible range (380-700 nm).
    
    Parameters
    ----------
    T : float
        Temperature in kelvin.
    
    Returns
    -------
    float
        Dimensionless fraction (0 to 1).
    """
    numerator, _ = quad(planck, lam_vis_min, lam_vis_max, args=(T,))
    denominator  = sigma_SB * T**4
    return numerator / denominator

# Compute f_vis for a range of temperatures (Table in the article)
temperatures_C = [700, 900, 1100, 1500]

print(f"{'T (°C)':>8}  {'T (K)':>8}  {'λ_max (nm)':>12}  {'f_vis (%)':>12}")
print("-" * 48)
for T_C in temperatures_C:
    T = T_C + 273.15
    fv = f_vis(T)
    lp = wien_peak(T)
    print(f"{T_C:>8}  {T:>8.2f}  {lp:>12.1f}  {fv*100:>12.4f}")

## 3. Question 1: Is the visible emission detectable by the human eye?

The human eye can detect light intensities as low as $\sim 10^{-10}$ W/m² under dark-adapted conditions (Hecht, *Optics*, 5th edn, 2017).

We estimate the visible power emitted per square metre of billet surface at 900 °C using:
$$P_{\rm vis} = f_{\rm vis}(T) \times \sigma_{\rm SB} T^4$$

In [ ]:
# Question 1: visible power at forge temperature
T = T_forge

# Total power per unit area (Stefan-Boltzmann)
P_total = sigma_SB * T**4

# Visible fraction
fv = f_vis(T)

# Visible power per unit area
P_vis = fv * P_total

# Detection threshold of the human eye
eye_threshold = 1e-10  # W/m²

print(f"Temperature: T = {T_forge_C} °C = {T:.2f} K")
print(f"Total emitted power:   P_total = {P_total:.3e} W/m²")
print(f"Visible fraction:      f_vis   = {fv*100:.4f} %")
print(f"Visible power:         P_vis   = {P_vis:.3e} W/m²")
print()
print(f"Human eye threshold:   {eye_threshold:.1e} W/m²")
print(f"Ratio P_vis / threshold = {P_vis/eye_threshold:.2e}")
print()
if P_vis > eye_threshold:
    print("ANSWER: Yes, the visible emission is detectable by the human eye.")
    print(f"It exceeds the detection threshold by a factor of {P_vis/eye_threshold:.1e}.")
else:
    print("ANSWER: No, the visible emission is not detectable by the human eye.")

## 4. Question 2: At what temperature does $f_{\rm vis}$ reach 1%?

We find this temperature by solving $f_{\rm vis}(T) = 0.01$ numerically using the Brent method.

In [ ]:
# Question 2: temperature at which f_vis = 1%
target = 0.01  # 1%

# We need to find T such that f_vis(T) - target = 0
# We bracket the solution between 1000 K and 5000 K
T_solution = brentq(lambda T: f_vis(T) - target, 1000, 5000)
T_solution_C = T_solution - 273.15

print(f"Temperature at which f_vis = 1%:")
print(f"  T = {T_solution:.1f} K = {T_solution_C:.1f} °C")
print(f"  Wien peak at this temperature: λ_max = {wien_peak(T_solution):.1f} nm")
print()
print("Verification:")
print(f"  f_vis({T_solution_C:.0f} °C) = {f_vis(T_solution)*100:.4f} %")

# Also show the trend table with finer resolution
print()
print("Trend table (finer resolution around the solution):")
print(f"{'T (°C)':>8}  {'f_vis (%)':>12}")
print("-" * 24)
for T_C in range(900, 1600, 100):
    T = T_C + 273.15
    print(f"{T_C:>8}  {f_vis(T)*100:>12.4f}")

## 5. Question 3: Which wavelength dominates in the visible range at 900 °C?

We plot $B(\lambda, T)$ over the visible range (380–700 nm) at $T = 1173$ K (900 °C) and identify the dominant wavelength. We also plot the full spectrum for context.

In [ ]:
# Question 3: dominant wavelength in the visible range at 900 °C
T = T_forge  # 1173.15 K

# Wavelength arrays
lam_vis  = np.linspace(380e-9, 700e-9, 1000)  # visible range
lam_full = np.linspace(200e-9, 5000e-9, 5000)  # full spectrum

# Planck distribution
B_vis  = planck(lam_vis,  T)
B_full = planck(lam_full, T)

# Dominant wavelength in visible range
idx_max_vis = np.argmax(B_vis)
lam_dom_vis = lam_vis[idx_max_vis] * 1e9  # convert to nm

print(f"Temperature: T = {T_forge_C} °C = {T:.2f} K")
print(f"Wien peak (full spectrum): λ_max = {wien_peak(T):.1f} nm (near infrared)")
print(f"Dominant wavelength in visible range: λ_dom = {lam_dom_vis:.1f} nm")
print()
if lam_dom_vis > 620:
    colour = "red"
elif lam_dom_vis > 590:
    colour = "orange"
elif lam_dom_vis > 560:
    colour = "yellow"
elif lam_dom_vis > 500:
    colour = "green"
elif lam_dom_vis > 450:
    colour = "blue"
else:
    colour = "violet"
print(f"Perceived colour: {colour} ({lam_dom_vis:.1f} nm)")
print(f"This is consistent with the orange-red glow depicted by Velázquez.")

# ---- Figure 1: Planck distribution in the visible range ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel (a): visible range only
ax = axes[0]
ax.plot(lam_vis * 1e9, B_vis, 'k-', linewidth=2)
ax.axvline(lam_dom_vis, color='red', linestyle='--', 
           label=f'Dominant: {lam_dom_vis:.0f} nm')
ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel(r'$B(\lambda, T)$ (W m$^{-2}$ m$^{-1}$)', fontsize=12)
ax.set_title(f'Planck distribution in the visible range\n'
             f'T = {T_forge_C} °C = {T:.0f} K', fontsize=11)
ax.legend(fontsize=11)
ax.set_xlim(380, 700)
ax.grid(True, alpha=0.3)

# Add colour background
colours_rgb = [
    (380, 450, '#8B00FF'),  # violet
    (450, 495, '#0000FF'),  # blue
    (495, 560, '#00FF00'),  # green
    (560, 590, '#FFFF00'),  # yellow
    (590, 620, '#FF7F00'),  # orange
    (620, 700, '#FF0000'),  # red
]
for lam_start, lam_end, col in colours_rgb:
    ax.axvspan(lam_start, lam_end, alpha=0.15, color=col)

# Panel (b): full spectrum with visible range highlighted
ax = axes[1]
ax.plot(lam_full * 1e9, B_full, 'k-', linewidth=2)
ax.axvspan(380, 700, alpha=0.2, color='yellow', label='Visible range')
ax.axvline(wien_peak(T), color='blue', linestyle='--',
           label=f'Wien peak: {wien_peak(T):.0f} nm')
ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel(r'$B(\lambda, T)$ (W m$^{-2}$ m$^{-1}$)', fontsize=12)
ax.set_title(f'Full Planck spectrum\nT = {T_forge_C} °C = {T:.0f} K', fontsize=11)
ax.legend(fontsize=11)
ax.set_xlim(200, 5000)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('planck_forge.pdf', dpi=150, bbox_inches='tight')
plt.savefig('planck_forge.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as planck_forge.pdf and planck_forge.png")

## 6. Summary of results

This cell collects the answers to the three questions in a compact format.

In [ ]:
# Summary
T = T_forge
fv = f_vis(T)
P_total = sigma_SB * T**4
P_vis = fv * P_total
T_1pct = brentq(lambda T: f_vis(T) - 0.01, 1000, 5000)

print("=" * 60)
print("SUMMARY OF RESULTS")
print("=" * 60)
print()
print(f"Forge temperature: {T_forge_C} °C = {T:.2f} K")
print(f"Wien peak wavelength: {wien_peak(T):.1f} nm (near infrared)")
print(f"Visible fraction f_vis: {fv*100:.4f} %")
print()
print("--- Question 1 ---")
print(f"Total emitted power:  {P_total:.3e} W/m²")
print(f"Visible power:        {P_vis:.3e} W/m²")
print(f"Eye threshold:        1e-10 W/m²")
print(f"→ Detectable: YES (exceeds threshold by {P_vis/1e-10:.1e})")
print()
print("--- Question 2 ---")
print(f"Temperature for f_vis = 1%: {T_1pct:.1f} K = {T_1pct-273.15:.1f} °C")
print()
print("--- Question 3 ---")
lam_vis  = np.linspace(380e-9, 700e-9, 1000)
B_vis    = planck(lam_vis, T)
lam_dom  = lam_vis[np.argmax(B_vis)] * 1e9
print(f"Dominant wavelength in visible range: {lam_dom:.1f} nm")
print(f"→ Orange-red, consistent with Velázquez's depiction")
print()
print("=" * 60)

## References

- Galeano J R (2026) *Art and Physics at the Museo del Prado*. European Journal of Physics.
- Hecht E (2017) *Optics*, 5th edn. Pearson.
- Young H D and Freedman R A (2015) *University Physics with Modern Physics*, 14th edn. Pearson.
- Planck M (1901) Ueber das Gesetz der Energieverteilung im Normalspectrum. *Annalen der Physik* **4** 553–563.

---
*Notebook by Javier R. Galeano, Universidad Politécnica de Madrid, 2026.*  
*Available at: https://github.com/jgaleano/prado-physics*  
*Licence: CC BY 4.0*